# Домашнє завдання №6 — TODO-агент з InMemoryStore

**Тема:** агенти та інструменти (LangChain / LangGraph v1).

**Мета:** створити агента, який додає, показує та видаляє завдання, зберігаючи їх у `InMemoryStore` між викликами агента.

У цьому ноутбуці:
- реалізовано інструменти `add_task`, `list_tasks`, `delete_task`;
- доступ до сховища робиться через `get_store()`;
- ID завдань генеруються через `uuid.uuid4()`;
- агент працює українською та проходить 5 тестових сценаріїв.

In [1]:
# Встановлення залежностей
%pip install -U langchain langgraph langchain-openai

  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.2.18
    Uninstalling langgraph-1.2.18:
      Successfully uninstalled langgraph-1.2.18
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [langgraph]


In [2]:
import os
import uuid
from datetime import datetime, timezone
from getpass import getpass

from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.config import get_store
from langgraph.store.memory import InMemoryStore


# Безпечне завантаження API ключа: env -> Colab Secrets -> ручне введення
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    try:
        from google.colab import userdata

        colab_key = userdata.get("OPENAI_API_KEY")
        if colab_key:
            api_key = colab_key
            os.environ["OPENAI_API_KEY"] = colab_key
    except Exception:
        pass

if not api_key:
    entered_key = getpass("Введіть OPENAI_API_KEY: ").strip()
    if entered_key:
        os.environ["OPENAI_API_KEY"] = entered_key
        api_key = entered_key

if not api_key:
    raise ValueError("OPENAI_API_KEY не встановлено. Додайте ключ у середовище або Colab Secrets.")

print("OPENAI_API_KEY: OK")

OPENAI_API_KEY: OK


In [4]:
TASKS_NAMESPACE = ("tasks",)


def _now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


@tool
def add_task(title: str) -> str:
    """Додає нове завдання та повертає підтвердження з ID."""
    task_title = title.strip()
    if not task_title:
        return "Не вдалося додати завдання: порожній текст."

    store = get_store()
    task_id = str(uuid.uuid4())
    store.put(
        TASKS_NAMESPACE,
        task_id,
        {
            "title": task_title,
            "created_at": _now_iso(),
        },
    )
    return f'Завдання "{task_title}" було успішно додано. ID: {task_id}'


@tool
def list_tasks() -> str:
    """Повертає список усіх збережених завдань."""
    store = get_store()
    items = list(store.search(TASKS_NAMESPACE, limit=1000))

    if not items:
        return "Список завдань порожній."

    # Стабільний порядок: спочатку раніше створені завдання.
    items.sort(key=lambda item: item.value.get("created_at", ""))

    lines = ["Ось всі ваші завдання:", ""]
    for idx, item in enumerate(items, 1):
        title = item.value.get("title", "(без назви)")
        lines.append(f"{idx}. {title} [{item.key}]")

    return "\n".join(lines)


@tool
def delete_task(task_id: str) -> str:
    """Видаляє завдання за точним ID."""
    target_id = task_id.strip()
    if not target_id:
        return "Не вказано ID для видалення."

    store = get_store()
    task = store.get(TASKS_NAMESPACE, target_id)
    if not task:
        return f"Завдання з ID {target_id} не знайдено."

    title = task.value.get("title", "(без назви)")
    store.delete(TASKS_NAMESPACE, target_id)
    return f'Завдання "{title}" було видалено. ID: {target_id}'

In [5]:
SYSTEM_PROMPT = """
Ти TODO-асистент, який відповідає українською мовою.

Правила роботи:
1) Для додавання завдання викликай інструмент add_task.
2) Для перегляду списку завжди викликай list_tasks.
3) Для видалення викликай delete_task за точним ID.
4) Якщо користувач просить видалити завдання за описом (наприклад, "видали завдання про хліб"),
   спочатку виклич list_tasks, знайди релевантний запис, візьми його повний ID і передай у delete_task.
5) Нічого не вигадуй: працюй тільки з тим, що є у сховищі.
6) Повертай короткі та зрозумілі відповіді для користувача.
""".strip()

store = InMemoryStore()
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

agent = create_agent(
    model=model,
    tools=[add_task, list_tasks, delete_task],
    system_prompt=SYSTEM_PROMPT,
    store=store,
)

print("Агент ініціалізовано.")

Агент ініціалізовано.


In [6]:
# Тестові сценарії з умови ДЗ

tests = [
    "Додай: купити хліб",
    "Додай: подзвонити лікарю",
    "Покажи всі завдання",
    "Видали завдання про хліб",
    "Що залишилось?",
]

for i, test in enumerate(tests, 1):
    print(f"\n{'=' * 50}")
    print(f"TEST {i}: {test}")
    print('=' * 50)

    result = agent.invoke({
        "messages": [
            {"role": "user", "content": test}
        ]
    })

    # Відповідь агента користувачу
    print(result["messages"][-1].content)

    # Для наочності покажемо, які інструменти були викликані
    tool_calls = []
    for message in result["messages"]:
        if getattr(message, "type", "") == "ai" and getattr(message, "tool_calls", None):
            tool_calls.extend(call.get("name", "unknown_tool") for call in message.tool_calls)
    if tool_calls:
        print(f"[tools] {', '.join(tool_calls)}")


TEST 1: Додай: купити хліб
Завдання "купити хліб" було успішно додано.
[tools] add_task

TEST 2: Додай: подзвонити лікарю
Завдання "подзвонити лікарю" було успішно додано.
[tools] add_task

TEST 3: Покажи всі завдання
Ось всі ваші завдання:

1. купити хліб [b1e2be72-5428-4589-b1c9-a07634040bf4]
2. подзвонити лікарю [59a3fa14-ea54-43df-a7d4-803533a26fea]
[tools] list_tasks

TEST 4: Видали завдання про хліб
Завдання "купити хліб" було успішно видалено.
[tools] list_tasks, delete_task

TEST 5: Що залишилось?
У вас залишилось одне завдання: подзвонити лікарю.
[tools] list_tasks


In [7]:
# Детермінована перевірка стану store без додаткового виклику LLM
remaining_items = list(store.search(TASKS_NAMESPACE, limit=1000))
remaining_titles = [item.value.get("title", "") for item in remaining_items]

print("\nПеревірка стану сховища:")
print("Кількість задач:", len(remaining_items))
print("Назви:", remaining_titles)

assert "подзвонити лікарю" in remaining_titles, "Очікувалось, що задача про лікаря залишиться."
assert "купити хліб" not in remaining_titles, "Задача про хліб мала бути видалена."

print("Перевірка пройдена: у сховищі лишилась задача про лікаря.")


Перевірка стану сховища:
Кількість задач: 1
Назви: ['подзвонити лікарю']
Перевірка пройдена: у сховищі лишилась задача про лікаря.


In [13]:
# Поспілкуємось із агентом у довільному режимі
while True:
    user_input = input("\nТвоя команда (exit — завершити): ")

    if user_input.lower() in ["exit", "quit", "вихід"]:
        print("Роботу завершено.")
        break

    response = agent.invoke({
        "messages": [
            {"role": "user", "content": user_input}
        ]
    })

    print("\nАгент:", response["messages"][-1].content)


Твоя команда (exit — завершити): Додай: Посміхайся і радій життю.

Агент: Завдання "Посміхайся і радій життю." було успішно додано.

Твоя команда (exit — завершити): Додай: Зателефонуй батькам і скажи, що любиш їх.

Агент: Завдання "Зателефонуй батькам і скажи, що любиш їх." було успішно додано.

Твоя команда (exit — завершити): Додай: Купити морську сіль для ванни.

Агент: Завдання "Купити морську сіль для ванни" було успішно додано.

Твоя команда (exit — завершити): Видали: сіль

Агент: Завдання "Купити морську сіль для ванни" було видалено.

Твоя команда (exit — завершити): Покажи список завдань прошу.

Агент: Ось ваш список завдань:

1. Посміхайся і радій життю. [3121c459-039a-4ad6-8950-25759727f3a7]
2. Зателефонуй батькам і скажи, що любиш їх. [26f80cce-e830-49e5-9992-60953fce6fa9]

Твоя команда (exit — завершити): Додай: Пам'ятай, що все відбувається по найкращому із можливих шляхів.

Агент: Завдання було успішно додано: "Пам'ятай, що все відбувається по найкращому із можливих ш

In [14]:
# Перевіримо, що агент пам'ятає
items = list(store.search(TASKS_NAMESPACE, limit=1000))

print(f"У сховищі {len(items)} завдань:\n")

for i, item in enumerate(items, 1):
    print(f"{i}. {item.value['title']}")
    print(f"   ID: {item.key}")
    print(f"   Створено: {item.value['created_at']}")

У сховищі 3 завдань:

1. Посміхайся і радій життю.
   ID: 3121c459-039a-4ad6-8950-25759727f3a7
   Створено: 2026-10-31T20:22:19.161178+00:00
2. Зателефонуй батькам і скажи, що любиш їх.
   ID: 26f80cce-e830-49e5-9992-60953fce6fa9
   Створено: 2026-10-31T20:22:40.357961+00:00
3. Пам'ятай, що все відбувається по найкращому із можливих шляхів.
   ID: 791d67e1-1b8d-4aaa-a0e8-7dd465862bbd
   Створено: 2026-10-31T20:24:47.930676+00:00


## Висновок і відповідність критеріям

- **Додавання завдань:** `add_task` використовує `get_store()`, генерує `uuid.uuid4()`, зберігає через `store.put()`.
- **Перегляд завдань:** `list_tasks` читає дані через `store.search()`, форматує з нумерацією та `ID`, обробляє порожній список.
- **Видалення завдань:** `delete_task` перевіряє існування через `store.get()`, видаляє через `store.delete()`, повертає інформативне повідомлення.
- **Агент:** створено через `create_agent(..., store=InMemoryStore())` з українським `system_prompt`.
- **Тестування:** додано 5 сценаріїв з умови та детерміновану перевірку фінального стану сховища. Додатково зроблено перевірку роботи агента на довільних питаннях у вільному режимі спілкування.